# Lab03 — Hugging Face: 10 giai đoạn trình bày và train đủ 10 epoch

Notebook gồm **10 cell code**, xen kẽ Markdown giải thích mục tiêu, module, output và câu hỏi thảo luận. Bật **GPU T4 + Internet**, chạy từ cell đầu đến cuối; sau restart phải chạy lại setup. Hai learning rate, mỗi candidate khởi tạo lại từ pretrained base, classification head mới, train đủ 10 epoch; không resume checkpoint cũ và không early stopping.

Augmentation chỉ thêm synthetic train, tối đa 25% của 8.530 mẫu gốc, số thực tế tùy câu hợp lệ. Validation/test giữ nguyên 1.066/1.066. Profile này chưa có kết quả GPU; không hứa giảm hết overfitting hoặc tăng accuracy. Các biến thể kế thừa label và cần kiểm tra ngữ nghĩa, không phải ground truth mới đã được gán nhãn thủ công.

| Cell code | Giai đoạn / quyết định | Module / file tham chiếu | Artifact và visualize |
|---|---|---|---|
| 1 | Clone, Python riêng, GPU | requirements.txt | revision, package/CUDA snapshot |
| 2 | Config, seed, mô hình, kế hoạch | config/augmented.json, config/augmented_study.json | bảng cấu hình; snapshot trong từng run |
| 3 | Preflight, sanity fixture | tests/ | outputs/preflight/: XML và log |
| 4 | Exercise 1: tokenizer, pipeline/manual | src/models/classifier.py, scripts/run_exercise1.py | predictions và 3 ảnh inline |
| 5 | Dataset và EDA trước tăng mẫu | scripts/preview_dataset.py | split counts, token histogram, mẫu train |
| 6 | Augmentation preview | scripts/preview_augmentation.py, src/datasets/augmentation.py | counts, before/after examples, parent records |
| 7 | Train đủ 10 epoch và đọc learning curves | scripts/run_study.py, src/training/trainer.py | console, epoch_history.json, 10-point curves |
| 8 | Giải thích checkpoint và threshold đã chọn | src/study.py, src/training/thresholds.py | validation table, frozen policy và threshold curves |
| 9 | Test và phân tích kết quả | src/workflow.py, src/study.py | confusion, ROC/PR, calibration, errors/inference |
| 10 | Audit và xuất ZIP | scripts/audit_study.py, scripts/package_study.py | review ZIP + checkpoint ZIP, SHA256 |

**Thứ tự tính toán:** `run_study.py` ở cell 7 thực hiện train, chốt model/threshold bằng validation, rồi test theo đúng thứ tự bên trong workflow. Cell 8–9 **chỉ đọc và trình bày** các quyết định/kết quả đã lưu, không chọn lại model sau khi xem test. Tách cell giúp trình bày, không thay thuật toán hoặc giả vờ test chạy trước selection.

Tham khảo [description](../description_lab_03.md), [walkthrough](../walkthrough_lab_03.md), [DL flow](../../../docs/dl_flow.md), [DL data](../../../docs/dl_data.md), [DL training](../../../docs/dl_training.md). Mọi hình hiển thị là PNG đã lưu dưới outputs/; JSON giữ records để audit/report.

## 1. Clone repository, môi trường riêng và GPU

Python kernel của Kaggle có thể khác Python của dependencies. Dùng venv Python 3.12 riêng và gọi child interpreter cho mọi xử lý ML; ảnh được hiển thị trong kernel cha. Repo tồn tại thì giữ nguyên kết quả, không reset/xóa dữ liệu. Nếu code cũ, cập nhật branch lab03 trước khi chạy. Chỉ GPU đầu tiên được dùng để giữ batch hiệu dụng ổn định.

In [ ]:
import os
import sys
import json
import shutil
import subprocess
import importlib.util
from pathlib import Path
from IPython.display import display, Image, Markdown, FileLink

REPO_URL = "https://github.com/lacthui06/Lab-DL.git"
BRANCH = "lab03"
REPO = Path("/kaggle/working/Lab-DL")
LAB = REPO / "labs/lab03"
ENV = Path("/kaggle/working/lab03-py312")
PYTHON = str(ENV / "bin/python")

assert shutil.which("nvidia-smi"), "Hãy bật Settings > Accelerator > GPU."
subprocess.run(["nvidia-smi"], check=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    print("Giữ repo và các kết quả hiện có.")
assert (LAB / "scripts/preview_dataset.py").is_file(), "Hãy cập nhật code lab03 phiên bản 10 cell."
subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.9.5"], check=True)
def uv(*args):
    subprocess.run([sys.executable, "-m", "uv", *map(str,args)], check=True)
uv("python", "install", "3.12")
if not Path(PYTHON).is_file(): uv("venv", "--python", "3.12", ENV)
uv("pip", "install", "--python", PYTHON, "--link-mode", "copy", "-r", LAB / "requirements.txt")
uv("pip", "check", "--python", PYTHON)

RUN_ENV = os.environ.copy()
RUN_ENV.update(CUDA_VISIBLE_DEVICES="0", TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1",
               CUBLAS_WORKSPACE_CONFIG=":4096:8", MPLCONFIGDIR="/kaggle/working/.mpl-lab03")
def run_python(*args, env=None):
    return subprocess.run([PYTHON,"-u",*map(str,args)], cwd=REPO,
                          env=RUN_ENV if env is None else env, check=True)
def selected_folder(kind):
    state=json.loads((LAB / "config/workflow.json").read_text())
    folder=(LAB / "outputs/runs" / state["active_runs"][kind]).resolve()
    assert folder.parent==(LAB / "outputs/runs").resolve(), "Run path không hợp lệ."
    return folder

def show_figures(folder, names=None):
    folder=Path(folder)
    paths=[folder/name for name in names] if names else sorted(folder.glob("*.png"))
    assert paths, f"Không có PNG trong {folder}"
    for path in paths:
        assert path.is_file(), f"Thiếu ảnh {path}"
        display(Markdown(f"**{path.name}**"))
        display(Image(filename=str(path)))

run_python("-c", "import sys,torch,transformers; print(sys.version); print(torch.__version__,transformers.__version__); assert torch.cuda.is_available(), 'GPU chưa khả dụng'; print(torch.cuda.get_device_name(0))")
free_gib=shutil.disk_usage("/kaggle/working").free/1024**3
print(f"Dung lượng trống: {free_gib:.1f} GiB")
assert free_gib>=6, "Cần ít nhất 6 GiB trống; hãy dùng session sạch."

## 2. Cấu hình, seed và kế hoạch thực nghiệm

Hai LR `[5e-6, 1e-5]`, weight decay 0.1, dropout encoder/attention 0.2, classifier 0.3. Giữ cross-entropy gốc, full official splits và 10 epoch/candidate. Seed 42 được workflow đặt trước model/data; config hiển thị dưới đây chính là file được snapshot vào từng run và gói ZIP.

Checkpoint chọn bằng validation Macro-F1; threshold chọn trên validation 19 điểm. Không chọn từ test. Tăng regularization có thể giảm học thuộc nhưng cũng có nguy cơ underfit. Hai candidate tổng 20 epoch, không phải một model học nối 20 epoch.

In [ ]:
import pandas as pd
CONFIG_FILE = LAB / "config/augmented.json"
STUDY_FILE = LAB / "config/augmented_study.json"
config=json.loads(CONFIG_FILE.read_text())
study_settings=json.loads(STUDY_FILE.read_text())
assert config["exercise2"]["num_train_epochs"]==10
assert study_settings["epochs_per_candidate"]==10
assert config["exercise2"]["eval_strategy"]=="epoch"
display(pd.DataFrame([{"seed":config["seed"], "backbone":config["exercise2"]["model_name"],
    "epochs_each":10, "learning_rates":str(study_settings["learning_rates"]),
    "batch_size":config["exercise2"]["batch_size"], "max_length":config["exercise2"]["max_length"],
    "weight_decay":config["exercise2"]["weight_decay"]}]))
display(config["exercise2"]["model_overrides"])
display(config["exercise2"]["augmentation"])

## 3. Preflight tests và sanity overfit

Tests dùng fixture cực nhỏ trên CPU để kiểm tra shape, loss/gradient, 10 epoch, augmentation và artifact. Không fine-tune full dataset trên CPU và không coi fixture là benchmark. Bước này phải pass trước training; XML và toàn bộ log được lưu trong outputs/preflight/.

In [ ]:
from datetime import datetime, timezone
validation_dir=LAB / "outputs/preflight" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
validation_dir.mkdir(parents=True,exist_ok=False)
TEST_ENV=RUN_ENV.copy(); TEST_ENV["CUDA_VISIBLE_DEVICES"]=""
with (validation_dir/"pytest.log").open("w",encoding="utf-8") as log:
    process=subprocess.Popen([PYTHON,"-u","-m","pytest","labs/lab03/tests","-q",
        f"--junitxml={validation_dir/'pytest.xml'}"],cwd=REPO,env=TEST_ENV,
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in process.stdout:
        print(line,end="");log.write(line)
    assert process.wait()==0, "Preflight failed; xem pytest.log, chưa chạy training."
print("Preflight PASSED:",validation_dir)

## 4. Exercise 1: tokenization và pretrained sentiment

Đối chiếu pipeline với manual forward/softmax trên cùng model và câu đầu vào. In token IDs, attention mask và probabilities; xem ảnh token heatmap và bảng so sánh. SST-2 pretrained phục vụ Exercise 1; Exercise 2 tải DistilBERT base và khởi tạo classification head mới.

In [ ]:
run_python("labs/lab03/scripts/run_exercise1.py")
EXERCISE1=selected_folder("exercise1")
EXERCISE1_ANALYSIS=selected_folder("exercise1_analysis")
display(pd.DataFrame(json.loads((EXERCISE1/"predictions.json").read_text())["manual"]))
display(json.loads((EXERCISE1/"tokenizer_breakdown.json").read_text()))
show_figures(EXERCISE1_ANALYSIS)

## 5. Full dataset và EDA trước augmentation

Xác nhận train/validation/test 8.530/1.066/1.066, nhãn nhị phân và phân bố lớp. Độ dài token chỉ đo train để giải thích max_length=128; mẫu minh họa cũng chỉ lấy train. Không fit tokenizer/vocabulary trên heldout. EDA được lưu thành run dataset_preview và đi cùng review ZIP nếu cùng dataset/tokenizer config.

In [ ]:
run_python("labs/lab03/scripts/preview_dataset.py","--config",CONFIG_FILE)
EDA=selected_folder("dataset_preview")
eda=json.loads((EDA/"data_preview.json").read_text())
display(pd.DataFrame([{"split":k,"samples":v,"negative":eda["label_counts"][k]["0"],
                      "positive":eda["label_counts"][k]["1"]} for k,v in eda["split_sizes"].items()]))
display(pd.DataFrame([eda["train_token_lengths"]]))
show_figures(EDA)

## 6. Augmentation: số mẫu và ví dụ trước/sau

Giữ mọi mẫu train gốc, thêm tối đa 25% synthetic variants. Thay một từ/cụm từ cùng cực tính, bỏ câu phủ định/tương phản, chặn trùng và cân bằng số mẫu thêm. Nhãn kế thừa có thể có noise: đọc bảng ví dụ trước khi tiếp tục. Không gọi synthetic variants là ground truth mới đã được gán nhãn thủ công.

Validation/test không tăng mẫu. Số thêm thực tế không nhất thiết đạt mức tối đa 2.132; không nhân bản nguyên xi để đủ quota. Baseline và Transformer dùng cùng augmented pool. Số step/epoch được tính theo effective train size, trong khi hình chính vẫn có 10 điểm epoch.

In [ ]:
run_python("labs/lab03/scripts/preview_augmentation.py","--config",CONFIG_FILE)
PREVIEW=selected_folder("augmentation_preview")
augmentation=json.loads((PREVIEW/"augmentation.json").read_text())
display(pd.DataFrame([{"official_train":augmentation["original_train_size"],
    "synthetic_added":augmentation["added_count"], "effective_train":augmentation["effective_train_size"],
    "validation":eda["split_sizes"]["validation"], "test":eda["split_sizes"]["test"]}]))
display(pd.DataFrame(augmentation["records"]).head(8))
show_figures(PREVIEW)

## 7. Train từ pretrained base: đủ 10 epoch mỗi candidate

Chạy cell này một lần cho mỗi study mới. Console in Epoch 01/10 đến Epoch 10/10 cho từng LR. Train Loss/Accuracy trong bảng chính đo train-eval tại cuối epoch, cùng eval mode với validation; optimization loss vẫn lưu riêng trong trainer_history.

Workflow còn chốt model/threshold và đánh giá test **bên trong** lệnh này theo đúng thứ tự. Cell 8–9 là nơi trình bày artifacts từng giai đoạn, không thực thi selection lại sau test. Checkpoint tốt nhất không bắt buộc ở epoch 10. Không resume checkpoint study cũ, không dùng sanity model làm model train chính. Chạy lại cell sẽ tạo study mới và tốn thêm GPU.

In [ ]:
run_python("labs/lab03/scripts/run_study.py","--config",CONFIG_FILE,"--settings",STUDY_FILE)
STUDY=selected_folder("study")
candidates=json.loads((STUDY/"candidate_comparison.json").read_text())
for candidate in candidates:
    folder=LAB / "outputs/runs" / candidate["run_id"]
    display(Markdown(f"### LR={candidate['learning_rate']:g} — đủ 10 epoch"))
    epoch_rows=json.loads((folder/"epoch_history.json").read_text())
    assert [r["epoch"] for r in epoch_rows]==list(range(1,11))
    display(pd.DataFrame(epoch_rows))
    show_figures(folder,["learning_curves.png"])
print("Study:",STUDY)

## 8. Trình bày checkpoint và threshold đã chốt bằng validation

Bảng dưới là quyết định đã được workflow ghi trước test: chọn checkpoint/LR bằng validation Macro-F1; hòa thì dùng validation loss và LR nhỏ hơn theo protocol. Từ checkpoint winner, quét threshold trên validation, ưu tiên Macro-F1 rồi gần 0.5 rồi threshold nhỏ hơn.

Không sửa tiêu chí sau khi nhìn test hoặc ép threshold khác 0.5. Với epoch schedule, checkpoint nằm ở cuối một epoch nguyên; cả quá trình vẫn chạy đủ 10 epoch. Biểu đồ so candidate theo epoch, tách với biểu đồ threshold theo giá trị ngưỡng.

In [ ]:
selection=json.loads((STUDY/"frozen_selection.json").read_text())
policy=json.loads((STUDY/"decision_threshold.json").read_text())
display(pd.DataFrame(candidates))
display(selection)
display(pd.DataFrame([{"policy":"Reference 0.5",**policy["baseline_metrics"]},
                      {"policy":"Validation selected",**policy["selected_metrics"]}]))
WINNER=LAB / "outputs/runs" / selection["winner"]["run_id"]
show_figures(STUDY,["candidate_validation_comparison.png"])
show_figures(WINNER,["validation_threshold/threshold_curves.png"])

## 9. Test, confidence và phân tích lỗi

Đây là metrics đã đo trên full test sau frozen selection. Xem cả Accuracy, Macro-F1, Precision/Recall, FP/FN và baseline; không chỉ lấy một số đẹp. ROC-AUC/AP đo ranking, confidence calibration là câu hỏi khác. CI bootstrap chỉ mô tả lấy mẫu test cho model cố định, chưa đo biến thiên giữa nhiều seed.

Đọc các câu sai như phủ định/tương phản/mỉa mai; giải thích ngôn ngữ là giả thuyết, không causal attribution. Mẫu confidence cao không đại diện ngẫu nhiên cho mọi lỗi. Test lịch sử đã từng được xem nên không gọi đây là blind benchmark hoàn toàn mới, cũng không dùng test để chọn lại model/ngưỡng.

In [ ]:
display(pd.DataFrame(json.loads((STUDY/"test_comparison.json").read_text())))
display(pd.DataFrame([json.loads((STUDY/"test_runtime_metrics.json").read_text())]))
display(json.loads((STUDY/"probability_diagnostics.json").read_text()))
show_figures(STUDY,["baseline_model_comparison.png","confusion_matrix.png",
    "test_threshold/confusion_matrix.png","metrics_bar.png","roc_pr_curves.png",
    "calibration_confidence.png","sample_predictions.png","inference_predictions.png"])
display(pd.DataFrame(json.loads((STUDY/"sample_predictions.json").read_text())))
display(pd.DataFrame(json.loads((STUDY/"inference_predictions.json").read_text())))

## 10. Audit, giữ notebook có output và tải ZIP

Audit tính lại selection/metrics từ records và đối chiếu manifests. Review ZIP gồm EDA, augmentation preview cùng cohort, candidates, study, console logs và preflight. Checkpoint ZIP giữ riêng best model/tokenizer/policy của winner; không commit weights lên Git thông thường.

Gốc ZIP nằm trong outputs/bundles/, được copy ra /kaggle/working để dễ tải. Refresh panel Output nếu chưa thấy. Save Version và File > Download Notebook để giữ output cell tương tác; script không tự thu hồi notebook đang mở từ trình duyệt. Sau chạy, report có thể dùng ngay PNG/bảng đã lưu, không cần train hoặc vẽ lại để lấy artifact.

In [ ]:
run_python("labs/lab03/scripts/audit_study.py","--study-run",STUDY)
run_python("labs/lab03/scripts/package_study.py","--study-run",STUDY)
bundle=json.loads((LAB / "outputs/bundles/latest_bundle.json").read_text())
for kind in ["review","checkpoint"]:
    if not bundle.get(kind):continue
    source=Path(bundle[kind]);download=Path("/kaggle/working")/source.name
    shutil.copy2(source,download)
    print(kind,":",download,f"({download.stat().st_size/1024**2:.1f} MB)")
    display(FileLink(str(download.relative_to(Path.cwd())) if download.is_relative_to(Path.cwd()) else str(download)))
print("SHA256 review:",bundle["review_sha256"])
print("Tải review ZIP để kiểm chứng/report; giữ riêng checkpoint ZIP.")